NoteBook 01 - Loading and Merging Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import glob
import re


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/AI_Trading_Dissertation"
FINAL_PROJECT_PATH = PROJECT_PATH + "/Final_Project"

RAW_DATA_PATH = FINAL_PROJECT_PATH + "/02_Data/Raw"
PROCESSED_DATA_PATH = FINAL_PROJECT_PATH + "/02_Data/Processed"
COMBINED_RAW_PATH = PROCESSED_DATA_PATH + "/Combined_Raw"
BENCHMARK_PROCESSED_PATH = PROCESSED_DATA_PATH + "/Benchmark"
RESULTS_PATH = FINAL_PROJECT_PATH + "/04_Results"

os.makedirs(COMBINED_RAW_PATH, exist_ok=True)
os.makedirs(BENCHMARK_PROCESSED_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)


#benchmark/index data handled separately in Part B below.
BENCHMARK_FOLDER_NAMES = ["Nifty50", "India VIX"]

print("Raw data folder:", RAW_DATA_PATH)


Raw data folder: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/02_Data/Raw


auto detect sectors and stocks

In [ ]:
sector_stock_map = {}

for sector in sorted(os.listdir(RAW_DATA_PATH)):
    if sector in BENCHMARK_FOLDER_NAMES:
        continue

    sector_path = os.path.join(RAW_DATA_PATH, sector)
    if not os.path.isdir(sector_path):
        continue

    stocks = [
        stock for stock in sorted(os.listdir(sector_path))
        if os.path.isdir(os.path.join(sector_path, stock))
    ]
    sector_stock_map[sector] = stocks

print("Sectors found:", len(sector_stock_map))
for sector, stocks in sector_stock_map.items():
    print(f"  {sector}: {stocks}")


Sectors found: 5
  Automobile: ['Eicher Motors', 'Mahindra & Mahindra']
  BANKING: ['HDFCB', 'ICICIBC']
  ENERGY: ['ONGC', 'Reliance']
  FMCG: ['Hindustan Unilever', 'ITC']
  IT: ['INFOSYS', 'TCS']


In [ ]:
def combine_stock_files(sector, stock):
    stock_folder = os.path.join(RAW_DATA_PATH, sector, stock)
    csv_files = sorted(glob.glob(os.path.join(stock_folder, "*.csv")))

    print(f"{sector} / {stock}: {len(csv_files)} files found")

    if len(csv_files) == 0:
        print(f"  WARNING: no CSV files found for {stock}")
        return None

    yearly_frames = []
    for file_path in csv_files:
        df = pd.read_csv(file_path)
        df["Stock"] = stock
        df["Sector"] = sector
        df["Source_File"] = os.path.basename(file_path)
        yearly_frames.append(df)

    combined = pd.concat(yearly_frames, ignore_index=True)
    return combined


merging all 10 stocks

In [ ]:
all_combined_data = {}
inventory_rows = []

for sector, stocks in sector_stock_map.items():
    for stock in stocks:
        print("\n" + "=" * 60)
        combined_df = combine_stock_files(sector, stock)

        if combined_df is None:
            inventory_rows.append({
                "Type": "Stock", "Sector": sector, "Name": stock,
                "Source_Files": 0, "Rows": 0, "Status": "NO FILES FOUND",
            })
            continue

        all_combined_data[stock] = combined_df

        safe_name = stock.replace(" ", "_").replace("&", "and")
        out_path = os.path.join(COMBINED_RAW_PATH, f"{safe_name}_All_Raw.csv")
        combined_df.to_csv(out_path, index=False)

        n_files = combined_df["Source_File"].nunique()
        inventory_rows.append({
            "Type": "Stock", "Sector": sector, "Name": stock,
            "Source_Files": n_files, "Rows": len(combined_df),
            "Status": "OK" if n_files == 11 else f"CHECK - only {n_files}/11 files",
        })
        print(f"  -> saved {len(combined_df)} rows from {n_files} files")

print("\nPart A complete —", len(all_combined_data), "stocks merged.")



Automobile / Eicher Motors: 11 files found
  -> saved 76732 rows from 11 files

Automobile / Mahindra & Mahindra: 11 files found
  -> saved 77229 rows from 11 files

BANKING / HDFCB: 11 files found
  -> saved 77222 rows from 11 files

BANKING / ICICIBC: 11 files found
  -> saved 77322 rows from 11 files

ENERGY / ONGC: 11 files found
  -> saved 77334 rows from 11 files

ENERGY / Reliance: 11 files found
  -> saved 77333 rows from 11 files

FMCG / Hindustan Unilever: 11 files found
  -> saved 77177 rows from 11 files

FMCG / ITC: 11 files found
  -> saved 77320 rows from 11 files

IT / INFOSYS: 11 files found
  -> saved 77326 rows from 11 files

IT / TCS: 11 files found
  -> saved 77300 rows from 11 files

Part A complete — 10 stocks merged.


Benchmarks: NIFTY 50 and India VIX

Searching for the raw files

In [ ]:
def peek_raw_file(folder_name):
    folder_path = os.path.join(RAW_DATA_PATH, folder_name)
    csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

    print(f"=== {folder_name} ===")
    print(f"Files found: {len(csv_files)}")
    for f in csv_files:
        print(" ", os.path.basename(f))

    if not csv_files:
        print("  WARNING: no CSV found - check the folder name/location in Drive.")
        return

    first_file = csv_files[0]
    print(f"\nRaw text, first 10 lines of {os.path.basename(first_file)}:")
    with open(first_file, "r") as f:
        for i, line in enumerate(f):
            print(" ", repr(line))
            if i >= 9:
                break


peek_raw_file("Nifty50")


=== Nifty50 ===
Files found: 1
  nifty50_2016_July_2026.csv

Raw text, first 10 lines of nifty50_2016_July_2026.csv:
  'Security,NIFTY Index,,,,,,,,,,,\n'
  'Start Date,01/01/2016 00:00,,,,,,,,,,,\n'
  'End Date,31/07/2026 00:00,,,,,,,,,,,\n'
  'Period,D,,,,,,,,,,,\n'
  'Currency,INR,,,,,,,,,,,\n'
  ',,,,,,,,,,,,\n'
  'Date,PX_LAST,Change,% Change,PX_OPEN,Change,% Change,PX_HIGH,Change,% Change,PX_LOW,Change,% Change\n'
  '31/07/2026,24383.6,66.45,0.273263931,24361.45,44.3,0.182175954,24429.4,112.25,0.461608371,24299.7,-17.45,-0.071760054\n'
  '30/07/2026,24317.15,66.95,0.276080197,24249.55,-0.65,-0.00268039,24342.95,92.75,0.382471072,24187.1,-63.1,-0.26020404\n'
  '29/07/2026,24250.2,264.85,1.104215698,24176.65,191.3,0.797570183,24283.55,298.2,1.243258906,24136.75,151.4,0.63121864\n'


In [ ]:
peek_raw_file("India VIX")


=== India VIX ===
Files found: 1
  invixn_2016_July_2026.csv

Raw text, first 10 lines of invixn_2016_July_2026.csv:
  'Security,INVIXN Index,,,,,,,,,,,\n'
  'Start Date,01/01/2016 00:00,,,,,,,,,,,\n'
  'End Date,31/07/2026 00:00,,,,,,,,,,,\n'
  'Period,D,,,,,,,,,,,\n'
  'Currency,INR,,,,,,,,,,,\n'
  ',,,,,,,,,,,,\n'
  'Date,PX_LAST,Change,% Change,PX_OPEN,Change,% Change,PX_HIGH,Change,% Change,PX_LOW,Change,% Change\n'
  '31/07/2026,11.755,-0.4,-3.29082682,12.155,0,0,12.21,0.055,0.452488688,11.645,-0.51,-4.195804196\n'
  '30/07/2026,12.155,0.1475,1.228398917,12.0075,0,0,12.3175,0.31,2.581719758,11.3025,-0.705,-5.871330418\n'
  '29/07/2026,12.0075,-0.555,-4.417910448,12.5625,0,0,12.5625,0,0,11.5625,-1,-7.960199005\n'


The loader, built for the confirmed format



In [ ]:
DATE_CANDIDATES = ["date", "dates"]
CLOSE_CANDIDATES = ["px_last", "close", "last price", "last", "price"]
OPEN_CANDIDATES = ["px_open", "open"]
HIGH_CANDIDATES = ["px_high", "high"]
LOW_CANDIDATES = ["px_low", "low"]


def find_column(df, candidates):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    return None


def find_header_row(file_path):
    # Bloomberg's HP export puts a few metadata lines before the real header -
    # finding the line that actually starts with "Date," instead of assuming a fixed count.
    with open(file_path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().lower().startswith("date,"):
                return i
    raise ValueError(f"Could not find a 'Date,...' header row in {file_path}")


def to_clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False), errors="coerce")


def load_benchmark(folder_name, output_name):
    folder_path = os.path.join(RAW_DATA_PATH, folder_name)
    csv_files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {folder_path}")

    frames = []
    for file_path in csv_files:
        header_row = find_header_row(file_path)
        df = pd.read_csv(file_path, skiprows=header_row)

        date_col = find_column(df, DATE_CANDIDATES)
        close_col = find_column(df, CLOSE_CANDIDATES)

        if date_col is None or close_col is None:
            raise ValueError(
                f"{file_path}: couldn't find a date and/or close column. "
                f"Columns present: {df.columns.tolist()}"
            )

        clean = pd.DataFrame()
        clean["date"] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        clean["close"] = to_clean_numeric(df[close_col])

        for label, candidates in [("open", OPEN_CANDIDATES), ("high", HIGH_CANDIDATES), ("low", LOW_CANDIDATES)]:
            col = find_column(df, candidates)
            if col is not None:
                clean[label] = to_clean_numeric(df[col])

        frames.append(clean)

    merged = pd.concat(frames, ignore_index=True)
    merged = merged.dropna(subset=["date", "close"])
    merged = merged.drop_duplicates(subset=["date"], keep="first")
    merged = merged.sort_values("date").reset_index(drop=True)  # fixes the newest-first source order

    print(f"{folder_name}: {len(merged)} rows, {merged['date'].min().date()} to {merged['date'].max().date()}")

    out_path = os.path.join(BENCHMARK_PROCESSED_PATH, f"{output_name}.csv")
    merged.to_csv(out_path, index=False)
    print(f"  -> saved to {out_path}")

    return merged


In [ ]:
nifty_df = load_benchmark("Nifty50", "NIFTY50_Daily")


Nifty50: 2621 rows, 2016-01-01 to 2026-07-31
  -> saved to /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/02_Data/Processed/Benchmark/NIFTY50_Daily.csv


In [ ]:
vix_df = load_benchmark("India VIX", "India_VIX_Daily")


India VIX: 2621 rows, 2016-01-01 to 2026-07-31
  -> saved to /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/02_Data/Processed/Benchmark/India_VIX_Daily.csv


In [ ]:
print("NIFTY 50 sample:")
display(nifty_df.head())
print("\nIndia VIX sample:")
display(vix_df.head())


NIFTY 50 sample:


,date,close,open,high,low
0,2016-01-01,7963.20,7938.45,7972.55,7909.80
1,2016-01-04,7791.30,7924.55,7937.55,7781.10
2,2016-01-05,7784.65,7828.40,7831.20,7763.25
3,2016-01-06,7741.00,7788.05,7800.95,7721.20
4,2016-01-07,7568.30,7673.35,7674.95,7556.60



India VIX sample:


,date,close,open,high,low
0,2016-01-01,14.2600,13.8700,14.3475,13.1050
1,2016-01-04,16.8350,14.2600,17.1025,14.0300
2,2016-01-05,16.7025,16.8350,16.8350,15.5675
3,2016-01-06,16.5525,16.7025,17.1625,15.6075
4,2016-01-07,18.9600,16.5525,19.2550,16.4900


Combined inventory, everything in one place

In [ ]:
benchmark_rows = [
    {"Type": "Benchmark", "Sector": "-", "Name": "NIFTY 50", "Source_Files": len(glob.glob(os.path.join(RAW_DATA_PATH, "Nifty50", "*.csv"))), "Rows": len(nifty_df), "Status": "OK"},
    {"Type": "Benchmark", "Sector": "-", "Name": "India VIX", "Source_Files": len(glob.glob(os.path.join(RAW_DATA_PATH, "India VIX", "*.csv"))), "Rows": len(vix_df), "Status": "OK"},
]

full_inventory_df = pd.DataFrame(inventory_rows + benchmark_rows)

inventory_path = os.path.join(RESULTS_PATH, "Raw_Data_Inventory.csv")
full_inventory_df.to_csv(inventory_path, index=False)
print("Saved:", inventory_path)

full_inventory_df


Saved: /content/drive/MyDrive/AI_Trading_Dissertation/Final_Project/04_Results/Raw_Data_Inventory.csv


,Type,Sector,Name,Source_Files,Rows,Status
0,Stock,Automobile,Eicher Motors,11,76732,OK
1,Stock,Automobile,Mahindra & Mahindra,11,77229,OK
2,Stock,BANKING,HDFCB,11,77222,OK
3,Stock,BANKING,ICICIBC,11,77322,OK
4,Stock,ENERGY,ONGC,11,77334,OK
5,Stock,ENERGY,Reliance,11,77333,OK
6,Stock,FMCG,Hindustan Unilever,11,77177,OK
7,Stock,FMCG,ITC,11,77320,OK
8,Stock,IT,INFOSYS,11,77326,OK
9,Stock,IT,TCS,11,77300,OK
